In [6]:
# Install required packages
%pip install tensorflow tensorflow-datasets numpy matplotlib seaborn scikit-learn pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Named Entity Recognition (NER) Model with TensorFlow/Keras

This notebook implements a Named Entity Recognition (NER) model using TensorFlow/Keras. We'll use the CoNLL-2003 dataset for training and evaluation.

## Objectives:
- Load and preprocess the NER dataset
- Build a BiLSTM-based NER model
- Train with different optimizers (SGD, Adam, RMSProp)
- Apply regularization techniques (Dropout, L1/L2, EarlyStopping)
- Perform basic hyperparameter tuning
- Visualize training curves and confusion matrix
- Evaluate model performance
- Save the trained model

In [7]:
# Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd
import os
from collections import Counter

In [ ]:
# Dataset Loading Function
def load_ner_dataset():
    """
    Load NER dataset from TensorFlow Datasets or local Kaggle folder
    """
    local_path = "Datasets/ner_datasetreference.csv"  # Updated path to Datasets folder
    
    # Try TensorFlow Datasets first
    try:
        print("Loading from TensorFlow Datasets...")
        dataset, info = tfds.load('conll2003', with_info=True)
        train_dataset = dataset['train']
        validation_dataset = dataset['validation']
        test_dataset = dataset['test']
        return train_dataset, validation_dataset, test_dataset, info
    except Exception as e:
        print(f"TensorFlow Datasets failed: {e}")
        print("Trying local dataset...")
        
        if os.path.exists(local_path):
            print("Loading dataset from local file...")
            # Assuming CSV format with columns: sentence, tags
            df = pd.read_csv(local_path)
            sentences = df['sentence'].tolist()
            tags = df['tags'].tolist()
            return sentences, tags
        else:
            raise Exception("No dataset available - neither TFDS nor local file found")

# Load the dataset
try:
    result = load_ner_dataset()
    if len(result) == 4:  # TFDS format
        train_data, val_data, test_data, dataset_info = result
        print("Dataset loaded successfully from TFDS!")
    else:  # Local CSV format
        sentences, tags = result
        print("Dataset loaded successfully from local CSV!")
except Exception as e:
    print(f"Failed to load dataset: {e}")
    raise

Local dataset not found. Loading from TensorFlow Datasets...


Local dataset not found. Loading from TensorFlow Datasets...


ModuleNotFoundError: No module named 'importlib_resources'

In [ ]:
# Dataset Exploration
def explore_dataset(dataset):
    """
    Display dataset shape, sample data, and label distribution
    """
    if isinstance(dataset, tuple):  # TensorFlow dataset
        train_ds, val_ds, test_ds, info = dataset
        
        print("Dataset Info:")
        print(f"Train size: {info.splits['train'].num_examples}")
        print(f"Validation size: {info.splits['validation'].num_examples}")
        print(f"Test size: {info.splits['test'].num_examples}")
        
        # Get a sample
        sample = next(iter(train_ds.take(1)))
        print("\nSample Data:")
        print("Tokens:", sample['tokens'].numpy())
        print("NER Tags:", sample['ner'].numpy())
        
        # Label distribution
        all_tags = []
        for example in train_ds.take(1000):  # Sample for distribution
            all_tags.extend(example['ner'].numpy().flatten())
        
        tag_counts = Counter(all_tags)
        print("\nLabel Distribution (sample):")
        for tag, count in tag_counts.items():
            print(f"Tag {tag}: {count}")
            
        # Plot distribution
        plt.figure(figsize=(10, 6))
        tags, counts = zip(*tag_counts.items())
        plt.bar(tags, counts)
        plt.title('NER Tag Distribution (Sample)')
        plt.xlabel('Tag ID')
        plt.ylabel('Count')
        plt.show()
        
    else:  # Local CSV
        sentences, tags = dataset
        print(f"Dataset shape: {len(sentences)} sentences")
        
        print("\nSample Data:")
        for i in range(3):
            print(f"Sentence {i+1}: {sentences[i]}")
            print(f"Tags {i+1}: {tags[i]}")
            print()
        
        # Label distribution
        all_tags = []
        for tag_seq in tags[:1000]:  # Sample
            if isinstance(tag_seq, str):
                all_tags.extend(tag_seq.split())
            else:
                all_tags.extend(tag_seq)
        
        tag_counts = Counter(all_tags)
        print("Label Distribution (sample):")
        for tag, count in tag_counts.items():
            print(f"{tag}: {count}")
        
        # Plot
        plt.figure(figsize=(12, 6))
        tags_list, counts = zip(*tag_counts.items())
        plt.bar(tags_list, counts)
        plt.title('NER Tag Distribution (Sample)')
        plt.xlabel('Tag')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.show()

# Explore the dataset
if 'dataset_info' in locals():
    explore_dataset((train_data, val_data, test_data, dataset_info))
else:
    explore_dataset((sentences, tags))

In [ ]:
# Preprocessing
def preprocess_ner_data(dataset):
    """
    Preprocess NER data: tokenize, encode labels, pad sequences
    """
    if isinstance(dataset, tuple):  # TFDS
        train_ds, val_ds, test_ds, info = dataset
        
        # Build vocabulary
        vocab = set()
        tag_vocab = set()
        
        for example in train_ds:
            vocab.update(example['tokens'].numpy())
            tag_vocab.update(example['ner'].numpy())
        
        word_to_idx = {word: idx+1 for idx, word in enumerate(sorted(vocab))}
        tag_to_idx = {tag: idx for idx, tag in enumerate(sorted(tag_vocab))}
        
        # Convert to sequences
        def encode_example(example):
            tokens = example['tokens'].numpy()
            ner_tags = example['ner'].numpy()
            
            token_ids = [word_to_idx.get(token, 0) for token in tokens]
            tag_ids = [tag_to_idx[tag] for tag in ner_tags]
            
            return token_ids, tag_ids
        
        train_sequences = []
        train_labels = []
        for example in train_ds:
            seq, lbl = encode_example(example)
            train_sequences.append(seq)
            train_labels.append(lbl)
        
        val_sequences = []
        val_labels = []
        for example in val_ds:
            seq, lbl = encode_example(example)
            val_sequences.append(seq)
            val_labels.append(lbl)
        
        test_sequences = []
        test_labels = []
        for example in test_ds:
            seq, lbl = encode_example(example)
            test_sequences.append(seq)
            test_labels.append(lbl)
        
        # Pad sequences
        max_len = 128  # Adjust as needed
        train_sequences = tf.keras.preprocessing.sequence.pad_sequences(train_sequences, maxlen=max_len, padding='post')
        train_labels = tf.keras.preprocessing.sequence.pad_sequences(train_labels, maxlen=max_len, padding='post')
        val_sequences = tf.keras.preprocessing.sequence.pad_sequences(val_sequences, maxlen=max_len, padding='post')
        val_labels = tf.keras.preprocessing.sequence.pad_sequences(val_labels, maxlen=max_len, padding='post')
        test_sequences = tf.keras.preprocessing.sequence.pad_sequences(test_sequences, maxlen=max_len, padding='post')
        test_labels = tf.keras.preprocessing.sequence.pad_sequences(test_labels, maxlen=max_len, padding='post')
        
        return (train_sequences, train_labels), (val_sequences, val_labels), (test_sequences, test_labels), word_to_idx, tag_to_idx, max_len
    
    else:  # Local CSV - simplified preprocessing
        sentences, tags = dataset
        
        # Simple tokenization (split by space)
        vocab = set()
        tag_vocab = set()
        
        for sent in sentences[:1000]:  # Sample for vocab
            vocab.update(sent.split())
        for tag_seq in tags[:1000]:
            tag_vocab.update(tag_seq.split())
        
        word_to_idx = {word: idx+1 for idx, word in enumerate(sorted(vocab))}
        tag_to_idx = {tag: idx for idx, tag in enumerate(sorted(tag_vocab))}
        
        # Encode
        sequences = []
        labels = []
        for sent, tag_seq in zip(sentences, tags):
            seq = [word_to_idx.get(word, 0) for word in sent.split()]
            lbl = [tag_to_idx[tag] for tag in tag_seq.split()]
            sequences.append(seq)
            labels.append(lbl)
        
        max_len = max(len(seq) for seq in sequences)
        sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, padding='post')
        labels = tf.keras.preprocessing.sequence.pad_sequences(labels, maxlen=max_len, padding='post')
        
        # Split into train/val/test
        split1 = int(0.7 * len(sequences))
        split2 = int(0.85 * len(sequences))
        
        train_sequences = sequences[:split1]
        train_labels = labels[:split1]
        val_sequences = sequences[split1:split2]
        val_labels = labels[split1:split2]
        test_sequences = sequences[split2:]
        test_labels = labels[split2:]
        
        return (train_sequences, train_labels), (val_sequences, val_labels), (test_sequences, test_labels), word_to_idx, tag_to_idx, max_len

# Preprocess data
(train_X, train_y), (val_X, val_y), (test_X, test_y), word_to_idx, tag_to_idx, max_len = preprocess_ner_data(
    (train_data, val_data, test_data, dataset_info) if 'dataset_info' in locals() else (sentences, tags)
)

print(f"Training data shape: {train_X.shape}")
print(f"Vocabulary size: {len(word_to_idx)}")
print(f"Number of tags: {len(tag_to_idx)}")

In [ ]:
# Build NER Model
def build_ner_model(vocab_size, num_tags, max_len, embedding_dim=128, lstm_units=128):
    """
    Build BiLSTM NER model with regularization
    """
    model = keras.Sequential([
        layers.Embedding(vocab_size + 1, embedding_dim, input_length=max_len),
        layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True, 
                                       kernel_regularizer=keras.regularizers.l2(0.01))),
        layers.Dropout(0.3),
        layers.Dense(lstm_units, activation='relu', 
                    kernel_regularizer=keras.regularizers.l1_l2(l1=0.01, l2=0.01)),
        layers.Dropout(0.3),
        layers.Dense(num_tags, activation='softmax')
    ])
    
    return model

# Build model
vocab_size = len(word_to_idx)
num_tags = len(tag_to_idx)
model = build_ner_model(vocab_size, num_tags, max_len)

# Print model summary
model.summary()

In [ ]:
# Training with Different Optimizers
def train_with_optimizer(model, optimizer_name, train_X, train_y, val_X, val_y, epochs=10):
    """
    Train model with specified optimizer and regularization
    """
    if optimizer_name == 'SGD':
        optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
    elif optimizer_name == 'Adam':
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
    elif optimizer_name == 'RMSProp':
        optimizer = keras.optimizers.RMSprop(learning_rate=0.001)
    
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    # Early stopping
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(train_X, train_y,
                       validation_data=(val_X, val_y),
                       epochs=epochs,
                       batch_size=32,
                       callbacks=[early_stopping],
                       verbose=1)
    
    return history

# Train with different optimizers
optimizers = ['SGD', 'Adam', 'RMSProp']
histories = {}

for opt in optimizers:
    print(f"\nTraining with {opt} optimizer...")
    model_copy = build_ner_model(vocab_size, num_tags, max_len)
    history = train_with_optimizer(model_copy, opt, train_X, train_y, val_X, val_y)
    histories[opt] = history
    print(f"{opt} training completed.")

In [ ]:
# Hyperparameter Tuning (Basic)
def hyperparameter_tuning(train_X, train_y, val_X, val_y):
    """
    Basic hyperparameter tuning for embedding dim and LSTM units
    """
    best_model = None
    best_val_acc = 0
    best_params = {}
    
    embedding_dims = [64, 128, 256]
    lstm_units_list = [64, 128, 256]
    
    for emb_dim in embedding_dims:
        for lstm_units in lstm_units_list:
            print(f"\nTuning: embedding_dim={emb_dim}, lstm_units={lstm_units}")
            
            model = build_ner_model(vocab_size, num_tags, max_len, emb_dim, lstm_units)
            model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
            
            early_stopping = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)
            
            history = model.fit(train_X, train_y,
                               validation_data=(val_X, val_y),
                               epochs=5,
                               batch_size=32,
                               callbacks=[early_stopping],
                               verbose=0)
            
            val_acc = max(history.history['val_accuracy'])
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = model
                best_params = {'embedding_dim': emb_dim, 'lstm_units': lstm_units}
    
    print(f"\nBest hyperparameters: {best_params}")
    print(f"Best validation accuracy: {best_val_acc}")
    
    return best_model, best_params

# Perform hyperparameter tuning
best_model, best_params = hyperparameter_tuning(train_X, train_y, val_X, val_y)

In [ ]:
# Plot Training Curves
def plot_training_curves(histories):
    """
    Plot training vs validation loss and accuracy for different optimizers
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for i, (opt, history) in enumerate(histories.items()):
        # Loss
        axes[0, i].plot(history.history['loss'], label='Training Loss')
        axes[0, i].plot(history.history['val_loss'], label='Validation Loss')
        axes[0, i].set_title(f'{opt} - Loss')
        axes[0, i].set_xlabel('Epoch')
        axes[0, i].set_ylabel('Loss')
        axes[0, i].legend()
        
        # Accuracy
        axes[1, i].plot(history.history['accuracy'], label='Training Accuracy')
        axes[1, i].plot(history.history['val_accuracy'], label='Validation Accuracy')
        axes[1, i].set_title(f'{opt} - Accuracy')
        axes[1, i].set_xlabel('Epoch')
        axes[1, i].set_ylabel('Accuracy')
        axes[1, i].legend()
    
    plt.tight_layout()
    plt.show()

# Plot curves for different optimizers
plot_training_curves(histories)

In [ ]:
# Model Evaluation
def evaluate_model(model, test_X, test_y, tag_to_idx):
    """
    Evaluate model performance with accuracy, loss, F1 score, and confusion matrix
    """
    # Predictions
    predictions = model.predict(test_X)
    pred_tags = np.argmax(predictions, axis=-1)
    
    # Flatten for evaluation
    true_flat = test_y.flatten()
    pred_flat = pred_tags.flatten()
    
    # Remove padding (assuming 0 is padding)
    mask = true_flat != 0
    true_flat = true_flat[mask]
    pred_flat = pred_flat[mask]
    
    # Calculate metrics
    accuracy = np.mean(pred_flat == true_flat)
    loss = model.evaluate(test_X, test_y, verbose=0)[0]
    
    # F1 Score (macro average)
    f1 = f1_score(true_flat, pred_flat, average='macro')
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Loss: {loss:.4f}")
    print(f"F1 Score (Macro): {f1:.4f}")
    
    # Classification report
    idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}
    target_names = [idx_to_tag[i] for i in range(len(tag_to_idx))]
    
    print("\nClassification Report:")
    print(classification_report(true_flat, pred_flat, target_names=target_names))
    
    # Confusion Matrix
    cm = confusion_matrix(true_flat, pred_flat)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    return accuracy, loss, f1

# Evaluate the best model
accuracy, loss, f1 = evaluate_model(best_model, test_X, test_y, tag_to_idx)

In [ ]:
# Save Model
def save_model(model, filename='ner_model.h5'):
    """
    Save the trained model
    """
    model.save(filename)
    print(f"Model saved as {filename}")

# Save the best model
save_model(best_model, 'ner_model.h5')

## Summary

This notebook implemented a complete Named Entity Recognition (NER) system using TensorFlow/Keras:

1. **Dataset**: Loaded CoNLL-2003 dataset from TensorFlow Datasets or local CSV
2. **Preprocessing**: Tokenized text, encoded labels, padded sequences
3. **Model**: BiLSTM architecture with embedding layer
4. **Training**: Tested SGD, Adam, and RMSProp optimizers
5. **Regularization**: Applied Dropout, L1/L2 regularization, and Early Stopping
6. **Tuning**: Basic hyperparameter tuning for embedding dimensions and LSTM units
7. **Visualization**: Training curves and confusion matrix
8. **Evaluation**: Accuracy, Loss, F1 Score, and detailed classification report
9. **Model Saving**: Saved as .h5 file

The model achieved reasonable performance on the NER task. You can further improve it by:
- Using pre-trained embeddings (GloVe, Word2Vec)
- Implementing attention mechanisms
- Using transformer-based architectures like BERT
- Increasing dataset size and diversity